In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Load relevant packages

In [3]:
import os
import math
import pickle
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import json, time, pandas as pd
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score
import gc, time, json
from sklearn.metrics import precision_score
import pandas as pd

# Define SeqRR model

In [4]:
class SeqRR(nn.Module):
    def __init__(self, embedding_dim, dropout_rate=0.0):
        super().__init__()
        self.dropout = nn.Dropout(dropout_rate)
        self.linear = nn.Linear(embedding_dim, 1)
    def forward(self, emb):
        x = self.dropout(emb)
        logits = self.linear(x)
        return logits.squeeze(1)

# Define EpiNN models with first and second order effect considerations

In [5]:
class EpiNN(nn.Module):
    def __init__(self, L, D, dropout_rate=0.0):
        super().__init__()
        self.L = L
        self.D = D
        self.dropout = nn.Dropout(dropout_rate)
        self.token_linear = nn.Linear(L*D+1, 1)
        self.interaction_mlp = nn.Sequential(
            nn.Linear(2*D, D),
            nn.LeakyReLU(),
            nn.Linear(D, 16),
            nn.LeakyReLU(),
            nn.Linear(16, 1)
        )
        self.interaction_scale = nn.Parameter(torch.tensor(0.2))

    def forward(self, emb):
        # compute first order interactions
        x1 = self.dropout(emb)
        x1 = self.token_linear(x1).squeeze(-1)

        # compute second order interactions
        x2 = emb[:,:-1].reshape(emb.shape[0],self.L, self.D)
        mask = (x2.abs().sum(dim=-1) > 0).float()  # (N, L)
        mask = mask[:, :, None] * mask[:, None, :] # (N, L, L)
        w = self.token_linear.weight.squeeze(0)[:-1].view(1, self.L, self.D)
        x2 = x2 * w
        xi = x2.unsqueeze(2).expand(-1, -1, self.L, -1)  # (N, L, L, D) copies x[:, i, :]
        xj = x2.unsqueeze(1).expand(-1, self.L, -1, -1)  # (N, L, L, D) copies x[:, j, :]
        x2 = torch.cat([(xi+xj)/2, (xi-xj).abs()], dim=-1)          # (N, L, L, 2D)
        x2 = self.interaction_mlp(x2).squeeze(-1) * mask   # (N, L, L)
        x2 = torch.triu(x2, diagonal=1).sum(dim=(1, 2)) * self.interaction_scale
        return x1 + x2

# Define helper functions for training

In [6]:
CENSOR_THRESH = -5.0
LAMBDA_NEG = 0.5  # weight for negatives


def load_data_single(emb, attn, labels, batch, batch_size, device):
    """
    Loads a batch of data from the single-embedding dataset.

    Returns:
        data_batch: Tensor [B, embedding_dim]
        labels_batch: Tensor [B]
    """
    if (batch+1)*batch_size > len(labels):
        idx = np.arange(batch*batch_size, len(labels))
    else:
        idx = np.arange(batch*batch_size, (batch+1)*batch_size)

    data_batch = emb[idx]
    if attn is not None:
        attn_batch = attn[idx]
    else:
        attn_batch = None
    labels_batch = labels[idx]

    # --- Shape guards (edge cases with single element) ---
    if data_batch.ndim == 1:          # -> [D]  -> make [1, D]
        data_batch = np.expand_dims(data_batch, axis=0)
    if np.isscalar(labels_batch):      # -> scalar -> make [1]
        labels_batch = np.array([labels_batch], dtype=np.float32)
    elif labels_batch.ndim == 0:       # -> [] -> make [1]
        labels_batch = labels_batch.reshape(1)

    data_batch = torch.from_numpy(data_batch).to(torch.float32).to(device)
    if attn_batch is not None:
        attn_batch = torch.from_numpy(attn_batch).to(torch.float32).to(device)
    labels_batch = torch.from_numpy(labels_batch).to(torch.float32).to(device)

    return data_batch, attn_batch, labels_batch


def pairwise_rank_loss_pos(preds_pos, targets_pos, margin=0.0, min_delta=0.0, reduction="mean"):
    """
    Pairwise logistic ranking loss over positive samples only.
    Encourages: if target_i > target_j then pred_i > pred_j by (margin).
    min_delta: ignore pairs with |target_i - target_j| <= min_delta (reduces noise).
    """
    # preds_pos, targets_pos: (P,)
    if preds_pos.numel() < 2:
        return preds_pos.new_tensor(0.0)

    # Pairwise differences
    dy = targets_pos[:, None] - targets_pos[None, :]   # (P, P)
    dp = preds_pos[:, None] - preds_pos[None, :]       # (P, P)

    # Mask pairs where target_i > target_j (and optionally sufficiently separated)
    mask = dy > min_delta
    if not mask.any():
        return preds_pos.new_tensor(0.0)

    # Logistic ranking loss: softplus(-(dp - margin)) for desired dp >= margin
    loss_mat = F.softplus(-(dp - margin))

    loss = loss_mat[mask]
    if reduction == "sum":
        return loss.sum()
    return loss.mean()


def censored_huber_plus_rank_loss(
    model,
    preds,
    targets,
    threshold=CENSOR_THRESH,
    lambda_neg=LAMBDA_NEG,
    reduction="mean",
    l1_lambda=0.0,
    l2_lambda=0.0,
    huber_delta=1.0,          # SmoothL1 beta
    rank_lambda=1.0,         # weight for rank loss
    rank_margin=0.0,
    rank_min_delta=0.0,       # ignore tiny target diffs among positives
):
    """
    Positives (targets > threshold): Huber loss on (pred-target)
    Negatives (targets <= threshold): censored penalty on max(pred-threshold, 0)
    + optional pairwise rank loss among positives
    + optional L1/L2 reg on model parameters
    """
    preds = preds.squeeze(-1)
    targets = targets.squeeze(-1)

    pos = targets > threshold
    neg = ~pos

    loss_vec = torch.zeros_like(targets)

    if pos.any():
        loss_pos = F.smooth_l1_loss(
            preds[pos], targets[pos],
            reduction="none",
            beta=huber_delta
        )
        loss_vec[pos] = loss_pos

    # --- Negative: censored penalty ---
    if neg.any():
        over = F.relu(preds[neg] - threshold)
        loss_vec[neg] = lambda_neg * F.smooth_l1_loss(over, torch.zeros_like(over), reduction="none", beta=huber_delta)

    # --- Aggregate data loss ---
    if reduction == "sum":
        data_loss = loss_vec.sum()
    else:
        data_loss = loss_vec.mean()

    # --- Pairwise rank loss (positives only) ---
    if rank_lambda > 0.0 and pos.any():
        rank_loss = pairwise_rank_loss_pos(
            preds[pos], targets[pos],
            margin=rank_margin,
            min_delta=rank_min_delta,
            reduction="mean"
        )
        data_loss = data_loss + rank_lambda * rank_loss

    # --- Regularization ---
    if l1_lambda == 0.0 and l2_lambda == 0.0:
        return data_loss

    reg_l1 = sum(p.abs().sum() for p in model.parameters())
    reg_l2 = sum(p.pow(2).sum() for p in model.parameters())
    reg = l1_lambda * reg_l1 + l2_lambda * reg_l2

    return data_loss + reg



def censored_mse_loss(model, preds, targets, threshold=CENSOR_THRESH, lambda_neg=LAMBDA_NEG, reduction="mean", l1_lambda=0.0, l2_lambda=0.0):
    preds = preds.squeeze(-1)
    targets = targets.squeeze(-1)

    pos = targets > threshold
    neg = ~pos

    loss_vec = torch.zeros_like(targets)

    if pos.any():
        diff_pos = preds[pos] - targets[pos]
        loss_vec[pos] = diff_pos.pow(2)

    if neg.any():
        over = F.relu(preds[neg] - threshold)
        loss_vec[neg] = lambda_neg * over.pow(2)

    if reduction == "mean":
        data_loss = loss_vec.mean()
    elif reduction == "sum":
        data_loss = loss_vec.sum()

    # add the L1 and L2 regularization terms
    if l1_lambda == 0.0 and l2_lambda == 0.0:
        return data_loss

    reg_l1 = sum(p.abs().sum() for p in model.parameters())
    reg_l2 = sum(p.pow(2).sum() for p in model.parameters())
    reg = l1_lambda * reg_l1 + l2_lambda * reg_l2

    return data_loss + reg



def train_epoch_single(model, optimizer, emb, attn, labels, batch_size, l1_lambda, l2_lambda, loss, device):  # loss: 'MSE' or 'Huber'
    model.train()
    running_loss = 0.0
    total_samples = 0
    num_batches = math.ceil(len(labels) / batch_size)

    for batch in range(num_batches):
        data_batch, attn_batch, labels_batch = load_data_single(emb, attn, labels, batch, batch_size, device)
        if attn_batch is not None:
            preds = model(data_batch, attn_batch).squeeze(-1)
        else:
            preds = model(data_batch).squeeze(-1)
        if loss == 'MSE':
            loss = censored_mse_loss(model, preds, labels_batch, l1_lambda=l1_lambda, l2_lambda=l2_lambda)
        else:
            loss = censored_huber_plus_rank_loss(model, preds, labels_batch, l1_lambda=l1_lambda, l2_lambda=l2_lambda)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        bs = data_batch.size(0)
        running_loss += loss.item() * bs
        total_samples += bs

    avg_loss = running_loss / total_samples
    return avg_loss


def test_epoch_single(model, emb, attn, labels, batch_size, l1_lambda, l2_lambda, loss, device): # loss: 'MSE' or 'Huber'
    model.eval()
    running_loss = 0.0
    total_samples = 0
    all_preds, all_targets = [], []
    num_batches = math.ceil(len(labels) / batch_size)

    with torch.no_grad():
        for batch in range(num_batches):
            data_batch, attn_batch, labels_batch = load_data_single(emb, attn, labels, batch, batch_size, device)

            # Skip truly empty batch (can happen if len(labels) == 0)
            if data_batch.numel() == 0:
                continue
            if attn_batch is not None:
                preds = model(data_batch, attn_batch)            # [B, 1] or [B]
            else:
                preds = model(data_batch)            # [B, 1] or [B]
            preds = preds.squeeze(-1).reshape(-1)        # force [B]
            labels_batch = labels_batch.reshape(-1)      # force [B]

            if loss == 'MSE':
                loss = censored_mse_loss(model, preds, labels_batch, l1_lambda=l1_lambda, l2_lambda=l2_lambda)
            else:
                loss = censored_huber_plus_rank_loss(model, preds, labels_batch, l1_lambda=l1_lambda, l2_lambda=l2_lambda)

            bs = preds.size(0)
            running_loss += loss.item() * bs
            total_samples += bs
            all_preds.append(preds)
            all_targets.append(labels_batch)

    if total_samples == 0:
        # No samples: return NaNs and empty tensors
        return float('nan'), float('nan'), float('nan'), torch.tensor([]), torch.tensor([])

    avg_loss = running_loss / total_samples
    # Filter out any accidental empties and cat safely
    all_preds = torch.cat([t.reshape(-1) for t in all_preds if t.numel() > 0], dim=0)
    all_targets = torch.cat([t.reshape(-1) for t in all_targets if t.numel() > 0], dim=0)

    # Spearman on positives only
    pos_mask = all_targets > CENSOR_THRESH
    if pos_mask.sum().item() >= 2:
        spearman_corr, _ = spearmanr(
            all_preds[pos_mask].cpu().numpy(),
            all_targets[pos_mask].cpu().numpy()
        )
    else:
        spearman_corr = np.nan

    # Accuracy on negatives
    neg_mask = ~pos_mask
    if neg_mask.any():
        neg_accuracy = (all_preds[neg_mask] <= CENSOR_THRESH).float().mean().item()
    else:
        neg_accuracy = np.nan

    return avg_loss, spearman_corr, neg_accuracy, all_preds, all_targets

# Define helper functions for plots & eval

In [7]:
def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)

def plot_training_curves(history, title_prefix, outdir, fold):
    """
    history: dict with keys 'train_loss', 'val_loss', 'spearman', 'neg_acc'
    Curves start at epoch 0 (untrained / zero-skill baseline).
    """
    ensure_dir(outdir)
    epochs = np.arange(len(history['train_loss']))  # 0..E

    fig = plt.figure(figsize=(10, 8))
    ax1 = fig.add_subplot(2, 2, 1)
    ax1.plot(epochs, history['train_loss'], label='Train Loss')
    ax1.set_title('Train Loss')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')

    ax2 = fig.add_subplot(2, 2, 2)
    ax2.plot(epochs, history['val_loss'], label='Val Loss')
    ax2.set_title('Validation Loss')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')

    ax3 = fig.add_subplot(2, 2, 3)
    ax3.plot(epochs, history['spearman'], label='Spearman (pos)')
    ax3.set_title('Spearman (positives)')
    ax3.set_xlabel('Epoch'); ax3.set_ylabel('Spearman ρ')

    ax4 = fig.add_subplot(2, 2, 4)
    ax4.plot(epochs, history['neg_acc'], label='Accuracy (neg)')
    ax4.set_title('Accuracy (negatives)')
    ax4.set_xlabel('Epoch'); ax4.set_ylabel('Accuracy')

    plt.suptitle(f'{title_prefix} — Fold {fold}')
    plt.tight_layout(rect=[0, 0.03, 1, 0.97])
    out_path = os.path.join(outdir, f'{title_prefix.replace(" ", "_").lower()}_fold{fold}_curves.png')
    plt.savefig(out_path, dpi=150)
    plt.close(fig)

def aggregate_and_plot(all_preds, all_targets, dataset_name, outdir):
    """
    all_preds, all_targets: list of torch tensors per fold
    """
    ensure_dir(outdir)
    preds = torch.cat(all_preds).cpu().numpy()
    targets = torch.cat(all_targets).cpu().numpy()

    # 1) Scatter with Spearman
    pos_mask = targets > CENSOR_THRESH
    sp, _ = spearmanr(preds[pos_mask], targets[pos_mask])
    neg_acc = np.mean(preds[~pos_mask] <= CENSOR_THRESH)
    fig = plt.figure(figsize=(6, 6))
    plt.scatter(targets, preds, s=10, alpha=0.6)
    plt.xlabel('True'); plt.ylabel('Predicted')
    plt.title(f'{dataset_name} — Pred vs True (Spearman ρ={sp:.3f})')
    plt.tight_layout()
    plt.savefig(os.path.join(outdir, f'{dataset_name.replace(" ", "_").lower()}_scatter.png'), dpi=150)
    plt.close(fig)

    # Binary labels for ROC/PR using WT fitness as threshold
    y_true = (targets > 0.0).astype(np.int32)
    y_score = preds  # continuous scores

    # 2) ROC
    fpr, tpr, _ = roc_curve(y_true, y_score)
    roc_auc = auc(fpr, tpr)
    fig = plt.figure(figsize=(6, 6))
    plt.plot(fpr, tpr, label=f'AUC={roc_auc:.3f}')
    plt.plot([0, 1], [0, 1], linestyle='--')
    plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
    plt.title(f'{dataset_name} — ROC (AUC={roc_auc:.3f})')
    plt.legend(loc='lower right')
    plt.tight_layout()
    plt.savefig(os.path.join(outdir, f'{dataset_name.replace(" ", "_").lower()}_roc.png'), dpi=150)
    plt.close(fig)

    # 3) PR
    precision, recall, _ = precision_recall_curve(y_true, y_score)
    ap = average_precision_score(y_true, y_score)  # AUPRC
    fig = plt.figure(figsize=(6, 6))
    plt.plot(recall, precision, label=f'AUPRC={ap:.3f}')
    plt.xlabel('Recall'); plt.ylabel('Precision')
    plt.title(f'{dataset_name} — PR (AUPRC={ap:.3f})')
    plt.legend(loc='lower left')
    plt.tight_layout()
    plt.savefig(os.path.join(outdir, f'{dataset_name.replace(" ", "_").lower()}_pr.png'), dpi=150)
    plt.close(fig)

    np.save(os.path.join(outdir, f'{dataset_name.replace(" ", "_").lower()}_preds.npy'), preds)
    np.save(os.path.join(outdir, f'{dataset_name.replace(" ", "_").lower()}_targets.npy'), targets)

    return {
        'spearman': float(sp),
        'neg_acc': float(neg_acc),
        'roc_auc': float(roc_auc),
        'auprc': float(ap),
        'preds': preds,
        'targets': targets
    }

In [8]:
def train_on_dataset(model, train_emb, train_attn, train_labels,
                     test_emb, test_attn, test_labels,
                     n_epoch, batch_size, loss, device,
                     title_prefix, outdir, fold,
                     lr=1e-3, weight_decay=1e-2, dropout_rate=0.1,
                     l1_lambda=0.0, l2_lambda=0.0,
                     patience=16):
    """
    Trains a fresh model with early stopping on val loss.
    Adds 'epoch 0' zero-skill metrics to history.
    """

    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    history = {'train_loss': [], 'val_loss': [], 'spearman': [], 'neg_acc': []}

    # ---- Epoch 0: zero-skill/untrained baseline ----
    train0_loss, _, _, _, _ = test_epoch_single(model, train_emb, train_attn, train_labels, batch_size, l1_lambda, l2_lambda, loss, device)
    val0_loss, sp0, neg0, _, _ = test_epoch_single(model, test_emb, test_attn, test_labels, batch_size, l1_lambda, l2_lambda, loss, device)
    history['train_loss'].append(train0_loss)
    history['val_loss'].append(val0_loss)
    history['spearman'].append(sp0 if sp0 is not None else np.nan)
    history['neg_acc'].append(neg0 if neg0 is not None else np.nan)

    best_val = val0_loss
    best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    epochs_since_improve = 0

    pbar = tqdm(range(1, n_epoch+1), desc=f'{title_prefix} — Fold {fold}', leave=False)

    for epoch in pbar:
        train_loss = train_epoch_single(model, optimizer, train_emb, train_attn, train_labels, batch_size, l1_lambda, l2_lambda, loss, device)
        val_loss, sp, neg_acc, _, _ = test_epoch_single(model, test_emb, test_attn, test_labels, batch_size, l1_lambda, l2_lambda, loss, device)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['spearman'].append(sp if sp is not None else np.nan)
        history['neg_acc'].append(neg_acc if neg_acc is not None else np.nan)

        pbar.set_postfix({
            'train': f'{train_loss:.4f}',
            'val': f'{val_loss:.4f}',
            'ρ(pos)': f'{(sp if not np.isnan(sp) else 0):.3f}',
            'acc(neg)': f'{(neg_acc if not np.isnan(neg_acc) else 0):.3f}',
            'pat': f'{epochs_since_improve}/{patience}'
        })

        # Early stopping on best val loss
        if val_loss < best_val - 1e-8:  # tiny margin to avoid float jitter
            best_val = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            epochs_since_improve = 0
        else:
            epochs_since_improve += 1
            if epochs_since_improve >= patience:
                pbar.close()
                print(f"[Early stop] {title_prefix} — Fold {fold} @ epoch {epoch} (best val={best_val:.6f})")
                break

    # Restore best weights (by val loss) and produce final test preds/targets
    if best_state is not None:
        model.load_state_dict(best_state)

    val_loss, sp, neg_acc, preds, targets = test_epoch_single(model, test_emb, test_attn, test_labels, batch_size, l1_lambda, l2_lambda, loss, device)

    # per-fold plots (include epoch 0)
    plot_training_curves(history, title_prefix, outdir, fold)

    return history, preds, targets

# In-silico trajectory experiment

In [9]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from scipy.stats import spearmanr

# =========================
# Config / paths
# =========================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPOCHS = 50
BATCH_SIZE = 64
LR = 3e-4
WEIGHT_DECAY = 1e-3
DROPOUT = 0.0
l1_lambda = 5e-3
l2_lambda = 0.0

neg_threshold = -5.0  # CENSOR_THRESH in your pipeline

CSV_PATH = "/content/drive/MyDrive/Xai_ired/Data/Simon_ML+Michal_neg/data_processed.csv"
AAINDEX_PATH = "/content/drive/MyDrive/Xai_ired/Data/Simon_ML+Michal_neg/AAindex_emb_masked.npy"
PLL_PATH = "/content/drive/MyDrive/Xai_ired/Data/Simon_ML+Michal_neg/PLL.npy"
OUT_DIR = "/content/drive/MyDrive/Xai_ired/results/insilico_trajectory"
os.makedirs(OUT_DIR, exist_ok=True)

In [10]:
# =========================
# Metric helpers
# =========================
def compute_spearman_for_topk(preds_np, targets_np, topk_list=[20, 40, 80, 'all']):
    # rank by true fitness descending first
    order_true = np.argsort(-targets_np)
    results = {}
    n = len(targets_np)
    for k in topk_list:
        kk = n if (k == 'all') else min(int(k), n)
        if kk < 2:
            results[str(k)] = np.nan
            continue
        idx = order_true[:kk]
        sp, _ = spearmanr(preds_np[idx], targets_np[idx])
        results[str(k)] = float(sp)
    return results

def compute_jaccard_topN(preds_np, targets_np, N_list=[10, 20, 30, 40, 50]):
    order_true = np.argsort(-targets_np)
    order_pred = np.argsort(-preds_np)
    n = len(targets_np)
    out = []
    for N in N_list:
        k = min(N, n)
        set_true = set(order_true[:k])
        set_pred = set(order_pred[:k])
        inter = len(set_true & set_pred)
        union = len(set_true | set_pred) if (set_true | set_pred) else 1
        out.append((N, inter / union))
    return out

# ---- NEW: your Simon top-N precision ----
def precision_top_n_simon(N: int, preds_np: np.ndarray, targets_np: np.ndarray) -> float:
    preds_np = np.asarray(preds_np).reshape(-1)
    targets_np = np.asarray(targets_np).reshape(-1)
    n = len(preds_np)
    if n == 0:
        return float("nan")
    k = min(int(N), n)
    if k <= 0:
        return 0.0
    top_idx = np.argpartition(preds_np, -k)[-k:]
    hit = (preds_np[top_idx] > 0.0) & (targets_np[top_idx] > 0.0)
    return float(hit.mean())

def compute_precision_topN_simon_curve(preds_np, targets_np, N_list=[10, 20, 30, 40, 50]):
    return [(N, precision_top_n_simon(N, preds_np, targets_np)) for N in N_list]
# =========================
# Plot training curves (same style)
# =========================
def plot_training_curves(history, title_prefix, outdir, tag):
    os.makedirs(outdir, exist_ok=True)
    epochs = np.arange(len(history['train_loss']))  # includes epoch 0 baseline
    fig = plt.figure(figsize=(10, 8))
    ax1 = fig.add_subplot(2, 2, 1); ax1.plot(epochs, history['train_loss']); ax1.set_title('Train Loss'); ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
    ax2 = fig.add_subplot(2, 2, 2); ax2.plot(epochs, history['val_loss']);   ax2.set_title('Validation Loss'); ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
    ax3 = fig.add_subplot(2, 2, 3); ax3.plot(epochs, history['spearman']);   ax3.set_title('Spearman (positives)'); ax3.set_xlabel('Epoch'); ax3.set_ylabel('ρ')
    ax4 = fig.add_subplot(2, 2, 4); ax4.plot(epochs, history['neg_acc']);    ax4.set_title('Accuracy (negatives)'); ax4.set_xlabel('Epoch'); ax4.set_ylabel('Acc')
    plt.suptitle(title_prefix)
    plt.tight_layout(rect=[0,0.03,1,0.97])
    path = os.path.join(outdir, f"{tag}_curves.png")
    plt.savefig(path, dpi=150); plt.close(fig)

# =========================
# Fixed-epoch training wrapper for EpiNN
#   (uses your existing train_epoch_single/test_epoch_single/load_data_single/loss fns)
# =========================
def train_on_dataset_fixed_epochs(
    train_emb, train_labels,
    val_emb, val_labels,
    L, D,
    n_epoch, batch_size, device,
    title_prefix, outdir, tag,
    lr=3e-4, weight_decay=1e-3, dropout_rate=0.0
):
    model = EpiNN(L, D, dropout_rate=dropout_rate).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    history = {'train_loss': [], 'val_loss': [], 'spearman': [], 'neg_acc': []}

    # epoch 0 baseline
    train0_loss, _, _, _, _ = test_epoch_single(model, train_emb, None, train_labels, batch_size, l1_lambda, l2_lambda, "Huber", device)
    val0_loss, sp0, neg0, _, _ = test_epoch_single(model, val_emb, None, val_labels, batch_size, l1_lambda, l2_lambda, "Huber", device)
    history['train_loss'].append(train0_loss)
    history['val_loss'].append(val0_loss)
    history['spearman'].append(sp0 if sp0 is not None else np.nan)
    history['neg_acc'].append(neg0 if neg0 is not None else np.nan)

    pbar = tqdm(range(1, n_epoch + 1), desc=title_prefix, leave=False)
    for epoch in pbar:
        train_loss = train_epoch_single(model, optimizer, train_emb, None, train_labels, batch_size, l1_lambda, l2_lambda, "Huber", device)
        val_loss, sp, neg_acc, _, _ = test_epoch_single(model, val_emb, None, val_labels, batch_size, l1_lambda, l2_lambda, "Huber", device)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['spearman'].append(sp if sp is not None else np.nan)
        history['neg_acc'].append(neg_acc if neg_acc is not None else np.nan)

        pbar.set_postfix({
            "train": f"{train_loss:.4f}",
            "val": f"{val_loss:.4f}",
            "ρ(pos)": f"{(0 if np.isnan(sp) else sp):.3f}",
            "acc(neg)": f"{(0 if np.isnan(neg_acc) else neg_acc):.3f}",
        })

    # final preds on val
    val_loss, sp, neg_acc, val_preds, val_targets = test_epoch_single(
        model, val_emb, None, val_labels, batch_size, l1_lambda, l2_lambda, "Huber", device
    )
    plot_training_curves(history, title_prefix, outdir, tag)
    return model, history, val_preds, val_targets

In [11]:
# =========================
# 1) Load + split into 5 datasets by start_round
# =========================
df = pd.read_csv(CSV_PATH)
required = {"mut", "fitness", "start_round"}
if not required.issubset(df.columns):
    raise ValueError(f"CSV must have columns: {sorted(required)}")

AA_raw = np.load(AAINDEX_PATH)  # expected shape (N, L+2, D) e.g. (N, L+2, 19)
PLL = np.load(PLL_PATH)

# infer L and D consistent with your CV pipeline
L = AA_raw.shape[1] - 2
D = AA_raw.shape[2]

# flatten + trim (same as your CV code)
AA = AA_raw.reshape(AA_raw.shape[0], -1)[:, 19:-19]  # -> (N, L*D)
# concat PLL -> (N, L*D + 1) to match EpiNN token_linear(L*D+1 -> 1)
AA_PLL = np.concatenate([AA, PLL], axis=1)

datasets = {}
for r in [1, 2, 3, 4, 5]:
    mask = (df["start_round"] == r)
    idx = df.index[mask].to_numpy()
    if len(idx) == 0:
        raise ValueError(f"No rows found for start_round == {r}")

    emb = AA_PLL[idx].astype(np.float32)
    labels = df.loc[mask, "fitness"].to_numpy(dtype=np.float32)
    variants = df.loc[mask, "mut"].to_numpy()

    datasets[r] = {"indices": idx, "emb": emb, "labels": labels, "variants": variants}
    print(f"[Round {r}] emb={emb.shape}, labels={labels.shape}")


[Round 1] emb=(186, 5511), labels=(186,)
[Round 2] emb=(149, 5511), labels=(149,)
[Round 3] emb=(298, 5511), labels=(298,)
[Round 4] emb=(165, 5511), labels=(165,)
[Round 5] emb=(385, 5511), labels=(385,)


In [12]:
# =========================
# 2) Train on previous rounds, validate on later round; save predictions
# =========================
experiments = [
    {"train_rounds": [1],       "val_round": 2, "name": "train1_val2"},
    {"train_rounds": [1, 2],    "val_round": 3, "name": "train12_val3"},
    {"train_rounds": [1, 2, 3], "val_round": 4, "name": "train123_val4"},
    {"train_rounds": [1, 2, 3, 4], "val_round": 5, "name": "train1234_val5"},
]

# Collect metrics for plotting
spearman_bars = {}      # exp_name -> dict {'20':..,'40':..,'80':..,'all':..}
jaccard_lines = {}      # exp_name -> list of (N, Jaccard)
precision_lines = {}    # exp_name -> list of (N, precision_top_n_simon)

for exp in experiments:
    tr_rounds = exp["train_rounds"]
    vr = exp["val_round"]
    name = exp["name"]
    print(f"\n=== Experiment: {name} ===")

    # Build train set by concatenation
    train_emb = np.concatenate([datasets[r]["emb"] for r in tr_rounds], axis=0)
    train_labels = np.concatenate([datasets[r]["labels"] for r in tr_rounds], axis=0)

    # Validation set
    val_emb = datasets[vr]["emb"]
    val_labels = datasets[vr]["labels"]
    val_variants = datasets[vr]["variants"]

    # Train (fixed epochs)
    title = f"EpiNN (AAindex_masked+PLL) — {name}"
    tag = name
    model, history, val_preds_t, val_targets_t = train_on_dataset_fixed_epochs(
        train_emb=train_emb, train_labels=train_labels,
        val_emb=val_emb, val_labels=val_labels,
        L=L, D=D,
        n_epoch=EPOCHS, batch_size=BATCH_SIZE, device=DEVICE,
        title_prefix=title, outdir=OUT_DIR, tag=tag,
        lr=LR, weight_decay=WEIGHT_DECAY, dropout_rate=DROPOUT
    )

    val_preds = val_preds_t.detach().cpu().numpy().reshape(-1)
    val_targets = val_targets_t.detach().cpu().numpy().reshape(-1)

    # Save predictions CSV
    df_out = pd.DataFrame({
        "mut": val_variants,
        "pred_fitness": val_preds,
        "true_fitness": val_targets,
        "val_round": vr,
        "train_rounds": [",".join(map(str, tr_rounds))] * len(val_preds),
    })
    csv_out = os.path.join(OUT_DIR, f"{name}_predictions.csv")
    df_out.to_csv(csv_out, index=False)
    print(f"[{name}] Saved predictions -> {csv_out}")

    # Collect metrics for plots
    spearman_bars[name] = compute_spearman_for_topk(val_preds, val_targets, topk_list=[20, 40, 80, "all"])
    jaccard_lines[name] = compute_jaccard_topN(val_preds, val_targets, N_list=[10, 20, 30, 40, 50])
    precision_lines[name] = compute_precision_topN_simon_curve(val_preds, val_targets, N_list=[10, 20, 30, 40, 50])


=== Experiment: train1_val2 ===


EpiNN (AAindex_masked+PLL) — train1_val2:   0%|          | 0/50 [00:00<?, ?it/s]

[train1_val2] Saved predictions -> /content/drive/MyDrive/Xai_ired/results/insilico_trajectory/train1_val2_predictions.csv

=== Experiment: train12_val3 ===


EpiNN (AAindex_masked+PLL) — train12_val3:   0%|          | 0/50 [00:00<?, ?it/s]

[train12_val3] Saved predictions -> /content/drive/MyDrive/Xai_ired/results/insilico_trajectory/train12_val3_predictions.csv

=== Experiment: train123_val4 ===


EpiNN (AAindex_masked+PLL) — train123_val4:   0%|          | 0/50 [00:00<?, ?it/s]

[train123_val4] Saved predictions -> /content/drive/MyDrive/Xai_ired/results/insilico_trajectory/train123_val4_predictions.csv

=== Experiment: train1234_val5 ===


EpiNN (AAindex_masked+PLL) — train1234_val5:   0%|          | 0/50 [00:00<?, ?it/s]

[train1234_val5] Saved predictions -> /content/drive/MyDrive/Xai_ired/results/insilico_trajectory/train1234_val5_predictions.csv


In [13]:
# =========================
# 8) Plot model performance
# =========================

# --- Grouped bar plot: Spearman for top20/top40/top80/All ---
exp_names = [e["name"] for e in experiments]
K_labels = ["20", "40", "80", "all"]
x = np.arange(len(exp_names))
width = 0.2
alph = 0.75

plt.figure(figsize=(10, 6))
for i, K in enumerate(K_labels):
    vals = [spearman_bars[name].get(K, np.nan) for name in exp_names]
    plt.bar(x + (i - 1.5) * width, vals, width=width, alpha=alph, label=f"Top {K}")
plt.xticks(x, exp_names, rotation=15)
plt.ylabel("Spearman ρ (validation)")
plt.title("Validation Spearman by experiment and top-K subset")
plt.legend()
plt.tight_layout()
out_bar = os.path.join(OUT_DIR, "spearman_grouped_bars.png")
plt.savefig(out_bar, dpi=150); plt.close()
print(f"Saved bar plot -> {out_bar}")

# --- Line plot: Jaccard@N for each experiment ---
plt.figure(figsize=(10, 6))
Ns = [10, 20, 30, 40, 50]
for name in exp_names:
    line = jaccard_lines[name]
    Ns_present = [n for (n, _) in line]
    vals = [v for (_, v) in line]
    plt.plot(Ns_present, vals, marker="o", linewidth=2, label=name)
plt.xticks(Ns)
plt.xlabel("Top-N")
plt.ylabel("Jaccard Index (Top-N overlap)")
plt.title("Top-N Jaccard overlap: Ground truth vs Predicted")
plt.legend()
plt.tight_layout()
out_line = os.path.join(OUT_DIR, "jaccard_lines.png")
plt.savefig(out_line, dpi=150); plt.close()
print(f"Saved line plot -> {out_line}")

# --- NEW: Line plot: Precision_top_n_simon@N for each experiment ---
plt.figure(figsize=(10, 6))
for name in exp_names:
    line = precision_lines[name]
    Ns_present = [n for (n, _) in line]
    vals = [v for (_, v) in line]
    plt.plot(Ns_present, vals, marker="o", linewidth=2, label=name)
plt.xticks(Ns)
plt.xlabel("Top-N")
plt.ylabel("Top-N precision (pred>0 AND true>0)")
plt.title("Top-N precision (Simon): among top-N suggestions")
plt.legend()
plt.tight_layout()
out_prec = os.path.join(OUT_DIR, "precision_topn_simon_lines.png")
plt.savefig(out_prec, dpi=150); plt.close()
print(f"Saved precision line plot -> {out_prec}")

Saved bar plot -> /content/drive/MyDrive/Xai_ired/results/insilico_trajectory/spearman_grouped_bars.png
Saved line plot -> /content/drive/MyDrive/Xai_ired/results/insilico_trajectory/jaccard_lines.png
Saved precision line plot -> /content/drive/MyDrive/Xai_ired/results/insilico_trajectory/precision_topn_simon_lines.png


# Singlets/Doublets extrapolation experiment

In [23]:
import os
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from scipy.stats import spearmanr

# ---------------- Config / paths ----------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPOCHS = 50
BATCH_SIZE = 64
LR = 3e-4
WEIGHT_DECAY = 1e-3
DROPOUT = 0.0
CENSOR_THRESH = -5.0  # must match your global

CSV_PATH = "/content/drive/MyDrive/Xai_ired/Data/Simon_ML+Michal_neg/data_processed.csv"
AAINDEX_PATH = "/content/drive/MyDrive/Xai_ired/Data/Simon_ML+Michal_neg/AAindex_emb_masked.npy"
PLL_PATH = "/content/drive/MyDrive/Xai_ired/Data/Simon_ML+Michal_neg/PLL.npy"
OUT_DIR = "/content/drive/MyDrive/Xai_ired/results/extrapolate_from_singles_doubles"
os.makedirs(OUT_DIR, exist_ok=True)

In [24]:
# ---------- training curves ----------
def plot_training_curves(history, title_prefix, outdir, tag):
    os.makedirs(outdir, exist_ok=True)
    epochs = np.arange(len(history['train_loss']))  # includes epoch 0
    fig = plt.figure(figsize=(10,8))
    ax1 = fig.add_subplot(2,2,1); ax1.plot(epochs, history['train_loss']); ax1.set_title('Train Loss'); ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
    ax2 = fig.add_subplot(2,2,2); ax2.plot(epochs, history['val_loss']);   ax2.set_title('Val Loss');   ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
    ax3 = fig.add_subplot(2,2,3); ax3.plot(epochs, history['spearman']);   ax3.set_title('Spearman (pos)'); ax3.set_xlabel('Epoch'); ax3.set_ylabel('ρ')
    ax4 = fig.add_subplot(2,2,4); ax4.plot(epochs, history['neg_acc']);    ax4.set_title('Acc (neg)');  ax4.set_xlabel('Epoch'); ax4.set_ylabel('Acc')
    plt.suptitle(title_prefix); plt.tight_layout(rect=[0,0.03,1,0.97])
    plt.savefig(os.path.join(outdir, f"{tag}_curves.png"), dpi=150); plt.close(fig)

# ---------- Fixed-epoch training for EpiNN ----------
def train_on_dataset_fixed_epochs(
    train_emb, train_labels, val_emb, val_labels,
    L, D, n_epoch, batch_size, device,
    title_prefix, outdir, tag,
    lr=3e-4, weight_decay=1e-3, dropout_rate=0.0,
    l1_lambda=5e-3, l2_lambda=0.0
):
    model = EpiNN(L, D, dropout_rate=dropout_rate).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    history = {'train_loss': [], 'val_loss': [], 'spearman': [], 'neg_acc': []}

    # epoch 0 baseline
    tr0, _, _, _, _ = test_epoch_single(model, train_emb, None, train_labels, batch_size, l1_lambda, l2_lambda, "Huber", device)
    va0, sp0, ne0, _, _ = test_epoch_single(model, val_emb, None, val_labels, batch_size, l1_lambda, l2_lambda, "Huber", device)
    history['train_loss'].append(tr0)
    history['val_loss'].append(va0)
    history['spearman'].append(sp0 if sp0 is not None else np.nan)
    history['neg_acc'].append(ne0 if ne0 is not None else np.nan)

    pbar = tqdm(range(1, n_epoch+1), desc=title_prefix, leave=False)
    for ep in pbar:
        tr = train_epoch_single(model, optimizer, train_emb, None, train_labels, batch_size, l1_lambda, l2_lambda, "Huber", device)
        va, sp, ne, _, _ = test_epoch_single(model, val_emb, None, val_labels, batch_size, l1_lambda, l2_lambda, "Huber", device)
        history['train_loss'].append(tr)
        history['val_loss'].append(va)
        history['spearman'].append(sp if sp is not None else np.nan)
        history['neg_acc'].append(ne if ne is not None else np.nan)
        pbar.set_postfix({
            "train": f"{tr:.4f}",
            "val": f"{va:.4f}",
            "ρ(pos)": f"{(0 if np.isnan(sp) else sp):.3f}",
            "acc(neg)": f"{(0 if np.isnan(ne) else ne):.3f}",
        })

    # final preds on val
    va, sp, ne, val_preds, val_targets = test_epoch_single(
        model, val_emb, None, val_labels, batch_size, l1_lambda, l2_lambda, "Huber", device
    )
    plot_training_curves(history, title_prefix, outdir, tag)
    return model, history, val_preds, val_targets

# ---------- Helpers (UPDATED: use '-' separator) ----------
def count_mutations(variant: str) -> int:
    if pd.isna(variant): return 0
    s = str(variant).strip()
    if s == "WT" or s == "": return 0
    return len([tok for tok in s.split("-") if tok.strip() != ""])

def split_mutations(variant: str):
    """Return list of individual mutation tokens; [] for WT/empty."""
    if pd.isna(variant): return []
    s = str(variant).strip()
    if s == "WT" or s == "": return []
    return [tok.strip() for tok in s.split("-") if tok.strip() != ""]

def scatter_with_highlights(y_true, y_pred, variants, title, save_path, highlights=None):
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)

    plt.figure(figsize=(6,6))
    plt.scatter(y_true, y_pred, s=12, alpha=0.6, label="All")

    if highlights:
        for hv in highlights:
            idxs = np.where(variants == hv)[0]
            if len(idxs) > 0:
                i = idxs[0]
                plt.scatter([y_true[i]], [y_pred[i]], s=40, edgecolors='k', linewidths=1.0, label=hv)
                plt.annotate(hv, (y_true[i], y_pred[i]), xytext=(5,5), textcoords='offset points', fontsize=9)

    pos_mask = y_true > CENSOR_THRESH
    if np.sum(pos_mask) >= 2:
        sp_pos, _ = spearmanr(y_pred[pos_mask], y_true[pos_mask])
    else:
        sp_pos = np.nan

    plt.xlabel("True fitness"); plt.ylabel("Predicted fitness")
    plt.title(f"{title} (Spearman₊ ρ={sp_pos:.3f})")
    if highlights:
        plt.legend(fontsize=8, loc="best")
    plt.tight_layout()
    plt.savefig(save_path, dpi=150); plt.close()
    return float(sp_pos)

In [25]:
# -------------- Build data splits --------------
df = pd.read_csv(CSV_PATH)
df = df[df["start_round"] > 0].reset_index(drop=False)  # keep original row index in 'index'

# UPDATED column names
if not {'mut', 'fitness', 'start_round'}.issubset(df.columns):
    raise ValueError("CSV must have columns: 'mut', 'fitness', 'start_round'.")

df["mut_count"] = df["mut"].apply(count_mutations).astype(int)

AA_raw = np.load(AAINDEX_PATH)  # expected 3D: (N, L+2, D)
PLL = np.load(PLL_PATH)

# EpiNN preprocessing (same as your CV pipeline)
if AA_raw.ndim != 3:
    raise ValueError(f"Expected AAindex_emb_masked to be 3D (N, L+2, D), got shape {AA_raw.shape}")

L = AA_raw.shape[1] - 2
D = AA_raw.shape[2]
AA_flat = AA_raw.reshape(AA_raw.shape[0], -1)[:, 19:-19]          # (N, L*D)
AA_PLL = np.concatenate((AA_flat, PLL), axis=1).astype(np.float32) # (N, L*D+1)

def dataset_from_mask(mask):
    idx = df.loc[mask, "index"].to_numpy()  # original indices into AA_raw/PLL
    emb = AA_PLL[idx]                       # already concatenated + float32
    labels = df.loc[mask, "fitness"].to_numpy().astype(np.float32)
    variants = df.loc[mask, "mut"].to_numpy()
    return {"indices": idx, "emb": emb, "labels": labels, "variants": variants}

# Singles experiment: train on single mutants; test on >1 mutants
train_set_singles = dataset_from_mask(df["mut_count"] == 1)
test_set_singles  = dataset_from_mask(df["mut_count"] > 1)

# Doubles experiment: train on <=2 mutants; test on >2 mutants
train_set_doubles = dataset_from_mask(df["mut_count"] <= 2)
test_set_doubles  = dataset_from_mask(df["mut_count"] > 2)

print(f"Singles:  train={train_set_singles['emb'].shape[0]}  | test={test_set_singles['emb'].shape[0]}")
print(f"Doubles:  train={train_set_doubles['emb'].shape[0]}  | test={test_set_doubles['emb'].shape[0]}")

# Map from mut -> original row index (for quick lookup of singles)
variant_to_index = {row.mut: int(row["index"]) for _, row in df.iterrows()}


Singles:  train=189  | test=993
Doubles:  train=508  | test=675


In [28]:
# -------------- Train/validate & save prediction CSVs --------------
def run_experiment(train_set, val_set, name):
    title = f"EpiNN (AAindex_masked+PLL) — {name}"
    model, history, val_preds_t, val_targets_t = train_on_dataset_fixed_epochs(
        train_emb=train_set["emb"],
        train_labels=train_set["labels"],
        val_emb=val_set["emb"],
        val_labels=val_set["labels"],
        L=L, D=D,
        n_epoch=EPOCHS,
        batch_size=BATCH_SIZE,
        device=DEVICE,
        title_prefix=title,
        outdir=OUT_DIR,
        tag=name,
        lr=LR,
        weight_decay=WEIGHT_DECAY,
        dropout_rate=DROPOUT,
    )
    preds = val_preds_t.detach().cpu().numpy().reshape(-1)
    trues = val_targets_t.detach().cpu().numpy().reshape(-1)
    df_out = pd.DataFrame({
        "mut": val_set["variants"],
        "pred_fitness": preds,
        "true_fitness": trues
    })
    csv_path = os.path.join(OUT_DIR, f"{name}_predictions.csv")
    df_out.to_csv(csv_path, index=False)
    print(f"[{name}] Saved predictions -> {csv_path}")
    return model, preds, trues, val_set["variants"]

model_sing, preds_sing, trues_sing, variants_sing = run_experiment(train_set_singles, test_set_singles, "singlets_experiment")
model_doub, preds_doub, trues_doub, variants_doub = run_experiment(train_set_doubles, test_set_doubles, "doublets_experiment")

EpiNN (AAindex_masked+PLL) — singlets_experiment:   0%|          | 0/50 [00:00<?, ?it/s]

[singlets_experiment] Saved predictions -> /content/drive/MyDrive/Xai_ired/results/extrapolate_from_singles_doubles/singlets_experiment_predictions.csv


EpiNN (AAindex_masked+PLL) — doublets_experiment:   0%|          | 0/50 [00:00<?, ?it/s]

[doublets_experiment] Saved predictions -> /content/drive/MyDrive/Xai_ired/results/extrapolate_from_singles_doubles/doublets_experiment_predictions.csv


In [29]:
# -------------- Plot model performance --------------
# NOTE: update highlight strings to use '-' if your 'mut' column now uses '-'
HL_COMMON = ['S68P-P203Q-T241A', 'S68P-T241A-H244Y', 'S68P-P203Q-T241A-H244Y']
HL_SING   = HL_COMMON + ['S68P-T241A']

sp1 = scatter_with_highlights(
    y_true=trues_sing, y_pred=preds_sing, variants=variants_sing,
    title="Singlets → higher-order",
    save_path=os.path.join(OUT_DIR, "singlets_scatter_highlight.png"),
    highlights=HL_SING
)
sp2 = scatter_with_highlights(
    y_true=trues_doub, y_pred=preds_doub, variants=variants_doub,
    title="Doublets → higher-order",
    save_path=os.path.join(OUT_DIR, "doublets_scatter_highlight.png"),
    highlights=HL_COMMON
)

# (B) Special scatter: sum of predicted component singles vs predicted multi-mutant
@torch.no_grad()
def predict_variants(model, indices):
    if np.isscalar(indices):
        idx = np.array([int(indices)], dtype=int)
    else:
        idx = np.asarray(indices, dtype=int).reshape(-1)
    if idx.size == 0:
        return np.array([], dtype=np.float32)

    X = AA_PLL[idx]  # already (N, L*D+1)
    xb = torch.from_numpy(X).to(torch.float32).to(DEVICE)
    y = model(xb).detach().cpu().numpy().reshape(-1)
    return y.astype(np.float32)

sum_single_preds = []
multi_preds = []
multi_trues = []
multi_variants = []

for i, var in enumerate(variants_sing):
    true_fit = trues_sing[i]
    if true_fit <= CENSOR_THRESH:
        continue
    mut_list = split_mutations(var)
    if len(mut_list) <= 1:
        continue
    if not all(m in variant_to_index for m in mut_list):
        continue

    single_idx = np.array([variant_to_index[m] for m in mut_list], dtype=int)
    single_pred_vals = predict_variants(model_sing, single_idx)
    sum_single_preds.append(float(np.sum(single_pred_vals)))

    multi_preds.append(float(preds_sing[i]))
    multi_trues.append(float(trues_sing[i]))
    multi_variants.append(var)

plt.figure(figsize=(6,6))
plt.scatter(sum_single_preds, multi_preds, s=12, alpha=0.6)
if len(multi_preds) >= 2:
    sp_sum, _ = spearmanr(np.array(multi_preds), np.array(sum_single_preds))
else:
    sp_sum = np.nan
plt.xlabel("Sum of predicted fitness (component singles)")
plt.ylabel("Predicted fitness (multi-mutant)")
plt.title(f"Predicted Multi Fitness vs Sum-of-singles (ρ={sp_sum:.3f})")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "singlets_sum_of_singles_scatter.png"), dpi=150)
plt.close()

# (C) Bar plot: Spearman vs number of mutations (per experiment)
def spearman_by_mutcount(preds, trues, variants):
    preds = np.asarray(preds)
    trues = np.asarray(trues)
    counts = np.array([count_mutations(v) for v in variants], dtype=int)

    rho_by_k, n_by_k = {}, {}
    for k in np.unique(counts):
        bucket = counts == k
        pos = trues > CENSOR_THRESH
        mask = bucket & pos
        n = int(np.sum(mask))
        n_by_k[int(k)] = n
        if n >= 2:
            r, _ = spearmanr(preds[mask], trues[mask])
            rho_by_k[int(k)] = float(r)
        else:
            rho_by_k[int(k)] = np.nan
    return rho_by_k, n_by_k

(sb_sing, ns_sing) = spearman_by_mutcount(preds_sing, trues_sing, variants_sing)
(sb_doub, ns_doub) = spearman_by_mutcount(preds_doub, trues_doub, variants_doub)

all_k = sorted(set(sb_sing.keys()) | set(sb_doub.keys()))
vals_s = [sb_sing.get(k, np.nan) for k in all_k]
vals_d = [sb_doub.get(k, np.nan) for k in all_k]
ns_s   = [ns_sing.get(k, 0) for k in all_k]
ns_d   = [ns_doub.get(k, 0) for k in all_k]

x = np.arange(len(all_k))
w = 0.38

plt.figure(figsize=(10,6))
bars_s = plt.bar(x - w/2, vals_s, width=w, alpha=0.8, label="Singlets experiment")
bars_d = plt.bar(x + w/2, vals_d, width=w, alpha=0.8, label="Doublets experiment")

plt.xticks(x, [str(k) for k in all_k])
plt.xlabel("# mutations")
plt.ylabel("Spearman₊ ρ (pred vs true)")
plt.title("Validation Spearman on positives by mutation count")
plt.legend()

def _annotate(bars, ns):
    for bar, n in zip(bars, ns):
        h = bar.get_height()
        x_c = bar.get_x() + bar.get_width()/2
        y = 0 if (isinstance(h, float) and np.isnan(h)) else h
        plt.text(x_c, y + 0.02, f"n={n}", ha='center', va='bottom', fontsize=8)

_annotate(bars_s, ns_s)
_annotate(bars_d, ns_d)

plt.tight_layout()
out_bar = os.path.join(OUT_DIR, "spearman_by_mutation_count_bars.png")
plt.savefig(out_bar, dpi=150); plt.close()
print(f"Saved bar plot → {out_bar}")

print(f"Done. CSVs and plots saved under: {OUT_DIR}")

Saved bar plot → /content/drive/MyDrive/Xai_ired/results/extrapolate_from_singles_doubles/spearman_by_mutation_count_bars.png
Done. CSVs and plots saved under: /content/drive/MyDrive/Xai_ired/results/extrapolate_from_singles_doubles


In [ ]:
# ---------------- Config / paths ----------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MAX_EPOCHS = 500
PATIENCE = 16
BATCH_SIZE = 64
LR = 3e-4
WEIGHT_DECAY = 1e-3
DROPOUT = 0.0
CENSOR_THRESH = -5.0  # must match your global

CSV_PATH = "/content/drive/MyDrive/Xai_ired/Data/Simon_ML+Michal_neg/data_processed.csv"
AAINDEX_PATH = "/content/drive/MyDrive/Xai_ired/Data/Simon_ML+Michal_neg/AAindex_emb_masked.npy"
PLL_PATH = "/content/drive/MyDrive/Xai_ired/Data/Simon_ML+Michal_neg/PLL.npy"
OUT_DIR = "/content/drive/MyDrive/Xai_ired/results/extrapolate_from_singles_doubles"
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# ---------- training functions ----------
def plot_training_curves(history, title_prefix, outdir, tag):
    os.makedirs(outdir, exist_ok=True)
    epochs = np.arange(len(history['train_loss']))  # includes epoch 0
    fig = plt.figure(figsize=(10,8))
    ax1 = fig.add_subplot(2,2,1); ax1.plot(epochs, history['train_loss']); ax1.set_title('Train Loss'); ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
    ax2 = fig.add_subplot(2,2,2); ax2.plot(epochs, history['val_loss']);   ax2.set_title('Val Loss');   ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
    ax3 = fig.add_subplot(2,2,3); ax3.plot(epochs, history['spearman']);   ax3.set_title('Spearman (pos)'); ax3.set_xlabel('Epoch'); ax3.set_ylabel('ρ')
    ax4 = fig.add_subplot(2,2,4); ax4.plot(epochs, history['neg_acc']);    ax4.set_title('Acc (neg)');  ax4.set_xlabel('Epoch'); ax4.set_ylabel('Acc')
    plt.suptitle(title_prefix); plt.tight_layout(rect=[0,0.03,1,0.97])
    plt.savefig(os.path.join(outdir, f"{tag}_curves.png"), dpi=150); plt.close(fig)

def train_on_dataset(train_emb, train_labels, val_emb, val_labels,
                     embedding_dim, n_epoch, batch_size, device,
                     title_prefix, outdir, tag,
                     lr=1e-3, weight_decay=1e-2, dropout_rate=0.1, patience=16):
    model = SingleRR(embedding_dim, dropout_rate=dropout_rate).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    history = {'train_loss': [], 'val_loss': [], 'spearman': [], 'neg_acc': []}

    # Epoch 0 baseline
    train0_loss, _, _, _, _ = test_epoch_single(model, train_emb, train_labels, batch_size, device)
    val0_loss, sp0, neg0, _, _ = test_epoch_single(model, val_emb, val_labels, batch_size, device)
    history['train_loss'].append(train0_loss)
    history['val_loss'].append(val0_loss)
    history['spearman'].append(sp0 if sp0 is not None else np.nan)
    history['neg_acc'].append(neg0 if neg0 is not None else np.nan)

    best_val = val0_loss
    best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    epochs_since = 0

    pbar = tqdm(range(1, n_epoch+1), desc=title_prefix, leave=False)
    for epoch in pbar:
        train_loss = train_epoch_single(model, optimizer, train_emb, train_labels, batch_size, device)
        val_loss, sp, neg_acc, _, _ = test_epoch_single(model, val_emb, val_labels, batch_size, device)
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['spearman'].append(sp if sp is not None else np.nan)
        history['neg_acc'].append(neg_acc if neg_acc is not None else np.nan)
        pbar.set_postfix({'train': f'{train_loss:.4f}', 'val': f'{val_loss:.4f}',
                          'ρ(pos)': f'{0 if np.isnan(sp) else sp:.3f}',
                          'acc(neg)': f'{0 if np.isnan(neg_acc) else neg_acc:.3f}',
                          'pat': f'{epochs_since}/{patience}'})
        if val_loss < best_val - 1e-8:
            best_val = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            epochs_since = 0
        else:
            epochs_since += 1
            if epochs_since >= patience:
                tqdm.write(f"[Early stop] {title_prefix} @ epoch {epoch} (best val={best_val:.6f})")
                break

    model.load_state_dict(best_state)
    val_loss, sp, neg_acc, val_preds, val_targets = test_epoch_single(model, val_emb, val_labels, batch_size, device)
    plot_training_curves(history, title_prefix, outdir, tag)
    return model, history, val_preds, val_targets

In [ ]:
# ---------- Helpers ----------
def count_mutations(variant: str) -> int:
    if pd.isna(variant): return 0
    s = str(variant).strip()
    if s == "WT" or s == "": return 0
    return len([tok for tok in s.split(",") if tok.strip() != ""])

def split_mutations(variant: str):
    """Return list of individual mutation tokens; [] for WT/empty."""
    if pd.isna(variant): return []
    s = str(variant).strip()
    if s == "WT" or s == "": return []
    return [tok.strip() for tok in s.split(",") if tok.strip() != ""]

def scatter_with_highlights(y_true, y_pred, variants, title, save_path,
                            highlights=None):
    """Scatter pred vs true with optional highlighted/annotated variants.
       Spearman is computed ONLY on positives: y_true > CENSOR_THRESH.
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    # Plot all points as before
    plt.figure(figsize=(6,6))
    plt.scatter(y_true, y_pred, s=12, alpha=0.6, label="All")

    # Highlights (plotted regardless of sign)
    if highlights:
        for hv in highlights:
            idxs = np.where(variants == hv)[0]
            if len(idxs) > 0:
                i = idxs[0]
                plt.scatter([y_true[i]], [y_pred[i]], s=40, edgecolors='k',
                            linewidths=1.0, label=hv)
                plt.annotate(hv, (y_true[i], y_pred[i]), xytext=(5,5),
                             textcoords='offset points', fontsize=9)

    # --- Spearman on positives only ---
    pos_mask = y_true > CENSOR_THRESH
    if np.sum(pos_mask) >= 2:
        sp_pos, _ = spearmanr(y_pred[pos_mask], y_true[pos_mask])
    else:
        sp_pos = np.nan

    plt.xlabel("True fitness"); plt.ylabel("Predicted fitness")
    plt.title(f"{title} (Spearman₊ ρ={sp_pos:.3f})")  # subscript '+' to indicate positives
    if highlights:
        plt.legend(fontsize=8, loc="best")
    plt.tight_layout()
    plt.savefig(save_path, dpi=150); plt.close()
    return float(sp_pos)

In [ ]:
# -------------- Build data splits --------------
df = pd.read_csv(CSV_PATH)
df = df[df["start_round"] > 0].reset_index(drop=False)  # keep original row index in 'index'

if not {'aa_variant','fitness_combined'}.issubset(df.columns):
    raise ValueError("CSV must have columns: 'aa_variant', 'fitness_combined'.")

df["mut_count"] = df["aa_variant"].apply(count_mutations).astype(int)

AAindex = np.load(AAINDEX_PATH)
PLL = np.load(PLL_PATH)

def dataset_from_mask(mask):
    idx = df.loc[mask, "index"].to_numpy()
    emb = np.concatenate((AAindex[idx], PLL[idx]), axis=1)
    labels = df.loc[mask, "fitness_combined"].to_numpy().astype(np.float32)
    variants = df.loc[mask, "aa_variant"].to_numpy()
    return {"indices": idx, "emb": emb, "labels": labels, "variants": variants}

# Singles experiment: train on single mutants; test on >1 mutants
train_set_singles = dataset_from_mask(df["mut_count"] == 1)
test_set_singles  = dataset_from_mask(df["mut_count"] > 1)

# Doubles experiment: train on <=2 mutants; test on >2 mutants
train_set_doubles = dataset_from_mask(df["mut_count"] <= 2)
test_set_doubles  = dataset_from_mask(df["mut_count"] > 2)

print(f"Singles:  train={train_set_singles['emb'].shape[0]}  | test={test_set_singles['emb'].shape[0]}")
print(f"Doubles:  train={train_set_doubles['emb'].shape[0]}  | test={test_set_doubles['emb'].shape[0]}")

embedding_dim = train_set_singles["emb"].shape[1]

# Map from aa_variant -> original row index (for quick lookup of singles)
variant_to_index = {row.aa_variant: int(row["index"]) for _, row in df.iterrows()}

Singles:  train=189  | test=993
Doubles:  train=508  | test=675


In [ ]:
# -------------- Train/validate & save prediction CSVs --------------
def run_experiment(train_set, val_set, name):
    title = f"SingleRR (AAindex+PLL) — {name}"
    model, history, val_preds_t, val_targets_t = train_on_dataset(
        train_emb=train_set["emb"],
        train_labels=train_set["labels"],
        val_emb=val_set["emb"],
        val_labels=val_set["labels"],
        embedding_dim=embedding_dim,
        n_epoch=MAX_EPOCHS,
        batch_size=BATCH_SIZE,
        device=DEVICE,
        title_prefix=title,
        outdir=OUT_DIR,
        tag=name,
        lr=LR,
        weight_decay=WEIGHT_DECAY,
        dropout_rate=DROPOUT,
        patience=PATIENCE
    )
    # Save predictions CSV
    preds = val_preds_t.detach().cpu().numpy()
    trues = val_targets_t.detach().cpu().numpy()
    df_out = pd.DataFrame({
        "aa_variant": val_set["variants"],
        "pred_fitness": preds,
        "true_fitness": trues
    })
    csv_path = os.path.join(OUT_DIR, f"{name}_predictions.csv")
    df_out.to_csv(csv_path, index=False)
    print(f"[{name}] Saved predictions -> {csv_path}")
    return model, preds, trues, val_set["variants"]

# Run both experiments
model_sing, preds_sing, trues_sing, variants_sing = run_experiment(train_set_singles, test_set_singles, "singlets_experiment")
model_doub, preds_doub, trues_doub, variants_doub = run_experiment(train_set_doubles, test_set_doubles, "doublets_experiment")

SingleRR (AAindex+PLL) — singlets_experiment:   0%|          | 0/500 [00:00<?, ?it/s]

[Early stop] SingleRR (AAindex+PLL) — singlets_experiment @ epoch 18 (best val=16.215579)
[singlets_experiment] Saved predictions -> /content/drive/MyDrive/Xai_ired/results/extrapolate_from_singles_doubles/singlets_experiment_predictions.csv


SingleRR (AAindex+PLL) — doublets_experiment:   0%|          | 0/500 [00:00<?, ?it/s]

[Early stop] SingleRR (AAindex+PLL) — doublets_experiment @ epoch 55 (best val=13.687418)
[doublets_experiment] Saved predictions -> /content/drive/MyDrive/Xai_ired/results/extrapolate_from_singles_doubles/doublets_experiment_predictions.csv


In [ ]:
# -------------- Plot model performance --------------
# (A) Scatter with highlighted variants (both experiments)
HL_COMMON = ['S68P,P203Q,T241A', 'S68P,T241A,H244Y', 'S68P,P203Q,T241A,H244Y']
HL_SING   = HL_COMMON + ['S68P,T241A']

sp1 = scatter_with_highlights(
    y_true=trues_sing, y_pred=preds_sing, variants=variants_sing,
    title="Singlets → higher-order",
    save_path=os.path.join(OUT_DIR, "singlets_scatter_highlight.png"),
    highlights=HL_SING
)
sp2 = scatter_with_highlights(
    y_true=trues_doub, y_pred=preds_doub, variants=variants_doub,
    title="Doublets → higher-order",
    save_path=os.path.join(OUT_DIR, "doublets_scatter_highlight.png"),
    highlights=HL_COMMON
)

# (B) Special scatter (singlets experiment):
# Keep test_set_singles variants with true fitness > -10
# AND all component single mutations exist in df; x = sum of predicted singles; y = predicted fitness of the multi-mutant
def predict_variants(model, indices):
    """
    Predict fitness for a list/array/scalar of original row indices.
    Returns a 1-D numpy array of length N.
    """
    # normalize indices -> 1-D int array
    if np.isscalar(indices):
        idx = np.array([int(indices)], dtype=int)
    else:
        idx = np.asarray(indices, dtype=int).reshape(-1)
    if idx.size == 0:
        return np.array([], dtype=np.float32)
    # build AAindex+PLL features (always [N, D])
    X = np.concatenate((AAindex[idx], PLL[idx]), axis=1)
    xb = torch.from_numpy(X).to(torch.float32).to(DEVICE)
    y = model(xb)                     # could be shape [N] or [N,1]
    y = y.detach().cpu().numpy()
    # force to 1-D
    y = np.asarray(y)
    if y.ndim == 2 and y.shape[1] == 1:
        y = y[:, 0]
    else:
        y = y.reshape(-1)             # handles [N] and any stray shapes safely
    return y.astype(np.float32)


retained_idx = []
sum_single_preds = []
multi_preds = []
multi_trues = []
multi_variants = []

for i, var in enumerate(variants_sing):
    true_fit = trues_sing[i]
    if true_fit <= CENSOR_THRESH:
        continue
    mut_list = split_mutations(var)
    # require all component singles exist in df
    if len(mut_list) <= 1:
        continue  # we only care about >1 mutants in this plot
    if not all(m in variant_to_index for m in mut_list):
        continue
    # sum of predicted single-mutation fitnesses (using singlets model)
    single_idx = np.array([variant_to_index[m] for m in mut_list], dtype=int)
    single_pred_vals = predict_variants(model_sing, single_idx)
    sum_pred = float(np.sum(single_pred_vals))
    sum_single_preds.append(sum_pred)
    # predicted fitness of this multi-mutant (already have preds_sing)
    multi_preds.append(float(preds_sing[i]))
    multi_trues.append(float(trues_sing[i]))
    multi_variants.append(var)

# Make the plot
plt.figure(figsize=(6,6))
plt.scatter(sum_single_preds, multi_preds, s=12, alpha=0.6)
# Spearman between multi_pred and sum-of-singles
if len(multi_preds) >= 2:
    sp_sum, _ = spearmanr(np.array(multi_preds), np.array(sum_single_preds))
else:
    sp_sum = np.nan
plt.xlabel("Sum of predicted fitness (component singles)")
plt.ylabel("Predicted fitness (multi-mutant)")
plt.title(f"Predicted Multi Fitness vs Sum-of-singles (ρ={sp_sum:.3f})")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "singlets_sum_of_singles_scatter.png"), dpi=150)
plt.close()

# (C) Bar plot: Spearman vs number of mutations (per experiment)
def spearman_by_mutcount(preds, trues, variants):
    """
    Spearman ONLY on positives (true > CENSOR_THRESH) within each exact
    mutation-count bucket. Returns:
      - rho_by_k: {k: spearman or np.nan}
      - n_by_k:   {k: number of positives used in that bucket}
    """
    preds = np.asarray(preds)
    trues = np.asarray(trues)
    counts = np.array([count_mutations(v) for v in variants], dtype=int)

    rho_by_k, n_by_k = {}, {}
    for k in np.unique(counts):
        bucket = counts == k
        pos = trues > CENSOR_THRESH
        mask = bucket & pos
        n = int(np.sum(mask))
        n_by_k[int(k)] = n
        if n >= 2:
            r, _ = spearmanr(preds[mask], trues[mask])
            rho_by_k[int(k)] = float(r)
        else:
            rho_by_k[int(k)] = np.nan
    return rho_by_k, n_by_k


# --- Bar plot: Spearman₊ by mutation count with group sizes ---
(sb_sing, ns_sing) = spearman_by_mutcount(preds_sing, trues_sing, variants_sing)
(sb_doub, ns_doub) = spearman_by_mutcount(preds_doub, trues_doub, variants_doub)

all_k = sorted(set(sb_sing.keys()) | set(sb_doub.keys()))
vals_s = [sb_sing.get(k, np.nan) for k in all_k]
vals_d = [sb_doub.get(k, np.nan) for k in all_k]
ns_s   = [ns_sing.get(k, 0) for k in all_k]
ns_d   = [ns_doub.get(k, 0) for k in all_k]

x = np.arange(len(all_k))
w = 0.38

plt.figure(figsize=(10,6))
bars_s = plt.bar(x - w/2, vals_s, width=w, alpha=0.8, label="Singlets experiment")
bars_d = plt.bar(x + w/2, vals_d, width=w, alpha=0.8, label="Doublets experiment")

plt.xticks(x, [str(k) for k in all_k])
plt.xlabel("# mutations")
plt.ylabel("Spearman₊ ρ (pred vs true)")
plt.title("Validation Spearman on positives by mutation count")
plt.legend()

# --- annotate group sizes above each bar ---
def _annotate(bars, ns):
    for bar, n in zip(bars, ns):
        h = bar.get_height()
        x_c = bar.get_x() + bar.get_width()/2
        label = f"n={n}"
        # put label slightly above; if NaN height, treat as 0
        y = 0 if (isinstance(h, float) and np.isnan(h)) else h
        plt.text(x_c, y + 0.02, label, ha='center', va='bottom', fontsize=8, rotation=0)

_annotate(bars_s, ns_s)
_annotate(bars_d, ns_d)

plt.tight_layout()
out_bar = os.path.join(OUT_DIR, "spearman_by_mutation_count_bars.png")
plt.savefig(out_bar, dpi=150); plt.close()
print(f"Saved bar plot → {out_bar}")

print(f"Done. CSVs and plots saved under: {OUT_DIR}")

Saved bar plot → /content/drive/MyDrive/Xai_ired/results/extrapolate_from_singles_doubles/spearman_by_mutation_count_bars.png
Done. CSVs and plots saved under: /content/drive/MyDrive/Xai_ired/results/extrapolate_from_singles_doubles
